# Notebook 05: Dimensión de Consistencia

## Introducción

La tercera dimensión de calidad de datos es la **Consistencia**: asegurar que los datos sean coherentes entre diferentes campos, registros y fuentes.

**Duración**: 30 minutos
**Nivel**: Intermedio

### Objetivos:

1. Entender qué es consistencia
2. Validar tipos de datos
3. Validar relaciones entre columnas
4. Detectar inconsistencias lógicas

## ¿Qué es Consistencia?

**Consistencia** mide si los datos son coherentes entre sí y cumplen con reglas lógicas.

### Pregunta Clave:
> ¿Los datos relacionados tienen sentido juntos?

### Tipos de Consistencia:

1. **Consistencia de Tipo**: Mismo tipo de dato en toda la columna
2. **Consistencia de Formato**: Mismo formato (ej: fechas)
3. **Consistencia Lógica**: Relaciones entre campos (ej: fecha_inicio < fecha_fin)
4. **Consistencia Referencial**: Claves foráneas válidas

### Impacto de Negocio:

- **Errores de cálculo**: Tipos inconsistentes causan errores
- **Análisis imposible**: Formatos inconsistentes impiden agregaciones
- **Lógica de negocio violada**: Inconsistencias lógicas indican errores

In [ ]:
import great_expectations as gx
import pandas as pd
from datetime import datetime

# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")
print(f"Datos cargados: {len(df)} registros")

## 1. Consistencia de Tipo

In [ ]:
# Análisis de tipos
print("=" * 70)
print("ANÁLISIS DE TIPOS DE DATOS")
print("=" * 70)
print("\nTipos actuales:")
print(df.dtypes)

# Configurar contexto
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Suite de consistencia de tipo
suite = context.suites.add(gx.ExpectationSuite(name="consistencia_tipo"))

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="price", type_="float64",
        meta={"dimension": "Consistencia", "tipo": "Tipo de dato"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="quantity", type_="int64",
        meta={"dimension": "Consistencia", "tipo": "Tipo de dato"}
    )
)

suite.save()

val_def = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite, name="val_tipo")
)

resultado = val_def.run(batch_parameters={"dataframe": df})
print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")

## 2. Consistencia de Formato

In [ ]:
# Suite de formato
suite_formato = context.suites.add(gx.ExpectationSuite(name="consistencia_formato"))

# UUID consistente
suite_formato.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="order_id",
        regex=r"^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$",
        meta={"dimension": "Consistencia", "tipo": "Formato"}
    )
)

suite_formato.save()

val_def_formato = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_formato, name="val_formato")
)

resultado_formato = val_def_formato.run(batch_parameters={"dataframe": df})
print(f"¿Validación de formato exitosa?: {' SÍ' if resultado_formato.success else ' NO'}")

## 3. Consistencia Lógica

Validar relaciones lógicas entre campos.

In [ ]:
# Calcular total (price * quantity)
df['total_calculado'] = df['price'] * df['quantity']

print("Ejemplo de consistencia lógica:")
print("price * quantity debería ser positivo")
print(f"\nRegistros con total <= 0: {(df['total_calculado'] <= 0).sum()}")

# Suite de lógica
suite_logica = context.suites.add(gx.ExpectationSuite(name="consistencia_logica"))

# El total debe ser positivo
suite_logica.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="total_calculado",
        min_value=0.01,
        meta={"dimension": "Consistencia", "tipo": "Lógica"}
    )
)

suite_logica.save()

val_def_logica = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_logica, name="val_logica")
)

resultado_logica = val_def_logica.run(batch_parameters={"dataframe": df})
print(f"\n¿Validación lógica exitosa?: {' SÍ' if resultado_logica.success else ' NO'}")

##  Ejercicio

Crea validaciones para verificar que:
1. `order_date` no sea posterior a hoy
2. `product_category` sea consistente (sin variaciones de mayúsculas/minúsculas)

In [ ]:
# TU CÓDIGO AQUÍ
pass

## Generar Data Docs

In [ ]:
context.build_data_docs()
print(" Data Docs generados!")
context.open_data_docs()

##  Resumen

Has aprendido sobre **Consistencia**:

1.  Consistencia de tipo con `ExpectColumnValuesToBeOfType`
2.  Consistencia de formato con regex
3.  Consistencia lógica entre campos
